# pyaegean Stage E — export the joint model to the shipped ONNX artifact

Takes the 12-epoch checkpoint from Drive, exports one ONNX graph, quantizes to int8,
and applies the **quantization gate**: fp32 vs int8 are both evaluated on dev through
the package's own inference module (the exact code path `pip install pyaegean` users
run); int8 ships only if it costs ≤ 0.3 points on UPOS, LAS, and lemma.

Output: `grc-joint.tar.gz` (the GitHub release asset) + `gate-report.json`.
GPU is optional here (export + gate run fine on CPU, just slower).

In [ ]:
# 1 — install + fresh clone (onnx/onnxruntime added for the export + gate)
!pip install -q "git+https://github.com/ryanpavlicek/pyaegean" torch transformers numpy onnx onnxruntime tokenizers
!rm -rf pyaegean_repo
!git clone https://github.com/ryanpavlicek/pyaegean.git pyaegean_repo

In [ ]:
# 2 — mount Drive and locate the 12-epoch checkpoint
from google.colab import drive
drive.mount('/content/drive')
CKPT = "/content/drive/MyDrive/pyaegean-stage-dplus-12/full/model"
!ls -la {CKPT}

In [ ]:
# 3 — rebuild the dataset (the gate evaluates on the dev fold)
!python pyaegean_repo/training/build_full_dataset.py --with-extras

In [ ]:
# 4 — export + quantize + gate (watch the 'gate: drops ... shipping ...' line)
!python pyaegean_repo/training/export_onnx.py --checkpoint {CKPT} \
    --data-dir pyaegean_repo/training/data --out pyaegean_repo/training/out/export

In [ ]:
# 5 — save the asset + report to Drive
!mkdir -p "/content/drive/MyDrive/pyaegean-stage-e-export"
!cp pyaegean_repo/training/out/export/grc-joint.tar.gz "/content/drive/MyDrive/pyaegean-stage-e-export/"
!cp pyaegean_repo/training/out/export/gate-report.json "/content/drive/MyDrive/pyaegean-stage-e-export/"
!ls -la "/content/drive/MyDrive/pyaegean-stage-e-export"

## What to bring back

- `gate-report.json` — the fp32/int8 dev numbers, the gate decision, and the tar's
  **sha256**
- `grc-joint.tar.gz` — download it to the local machine (the maintainer-side step
  publishes it as the GitHub release asset, pins the URL + sha256 in the package, and
  re-measures every headline number through `pip`-installed pyaegean)